<br>

<h1 style="text-align:center;">NLP Features</h1>

<br>

## Initial Setup

---


In [1]:
# Import standard libraries
import re
import logging
from collections import Counter
from typing import List, Dict, Any, Union

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt

# NLP libraries
import contractions
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import spacy
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textstat import textstat

# Feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import KeyedVectors, Word2Vec
from sentence_transformers import SentenceTransformer

# Deep learning
import torch

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
# Download required NLTK data
nltk.download('all', quiet=True)

True

<br>

## Load Dataset

---


In [4]:
# Load the dataset 
df = pd.read_csv("./../Data/raw/groups.csv")[["text", "emotion"]]
df

,text,emotion
0,Did you kiss?,surprise
1,Yeah.,neutral
2,But no !,anger
3,Why do you have a problem?,surprise
4,Did you kiss?,surprise
...,...,...
24290,And tribes put everything on the line.,happiness
24291,I came in claiming that I had the most knowled...,happiness
24292,At no point during my time here was that dispr...,happiness
24293,There is nothing in the last few days of my ex...,happiness


In [5]:
# Set the name of the column to apply feature extraction on
column_name = 'text'

<br>

## Part-of-Speech (POS) Tagging

---

- Extract POS tags for each sentence in your transcription using an NLP library such as SpaCy or NLTK.

- Create a new column in your dataset named ‘POS_Tags’ and store the POS tags for each sentence.

In [6]:
# Define a function to extract Part-of-Speech tags from text using NLTK
def extract_pos_tags_nltk(text, language='english'):
    """
    Extract Part-of-Speech tags from a given text using NLTK.
    
    Args:
        text (str): Input text to process
        language (str): Language code ('english', 'spanish', etc.)
        
    Returns:
        list: List of tuples containing (word, POS tag)
    """
    # Check if input is a valid string and return empty list if not
    if not isinstance(text, str) or not text.strip():
        return []
    
    try:
        # Convert text to lowercase and remove leading/trailing whitespace
        text = text.lower().strip()
        
        # Split text into individual words/tokens using NLTK's word_tokenize function
        tokens = word_tokenize(text, language=language)
        
        # Extract first three characters of language name for language code
        language_code = language.lower()[:3]
        
        # For English language, use the standard POS tagger
        if language_code == 'eng':
            
            # Apply NLTK's default POS tagger for English
            pos_tags = nltk.pos_tag(tokens)
        
        # For other supported languages, use the universal tagset
        elif language_code in ['spa', 'fre', 'ger', 'ita', 'por', 'dut']:
            
            # Apply NLTK's universal POS tagger with appropriate language setting
            pos_tags = nltk.pos_tag(tokens, tagset='universal', lang=language_code)
        
        # For unsupported languages, use a basic fallback approach
        else:
            
            # Assign 'X' tag (universal tag for "other") to all tokens
            pos_tags = [(token, 'X') for token in tokens]
            
            # Print warning message about limited language support
            print(f"Warning: Full POS tagging not supported for {language}. Using basic tokenization only.")
        
        # Remove any empty tokens from the results
        pos_tags = [(word, tag) for word, tag in pos_tags if word.strip()]
        
        # Return the list of word-tag pairs
        return pos_tags
    
    # Handle any exceptions that might occur during processing
    except Exception as e:
        
        # Print error message with details
        print(f"Error in POS tagging: {str(e)}")
        
        # Return empty list instead of raising an exception
        return []

In [7]:
# Apply POS tagging to the DataFrame using NLTK
df['POS_Tags'] = df[column_name].apply(extract_pos_tags_nltk)

df

,text,emotion,POS_Tags
0,Did you kiss?,surprise,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]"
1,Yeah.,neutral,"[(yeah, NN), (., .)]"
2,But no !,anger,"[(but, CC), (no, DT), (!, .)]"
3,Why do you have a problem?,surprise,"[(why, WRB), (do, VBP), (you, PRP), (have, VBP..."
4,Did you kiss?,surprise,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]"
...,...,...,...
24290,And tribes put everything on the line.,happiness,"[(and, CC), (tribes, VB), (put, VBN), (everyth..."
24291,I came in claiming that I had the most knowled...,happiness,"[(i, NN), (came, VBD), (in, IN), (claiming, VB..."
24292,At no point during my time here was that dispr...,happiness,"[(at, IN), (no, DT), (point, NN), (during, IN)..."
24293,There is nothing in the last few days of my ex...,happiness,"[(there, EX), (is, VBZ), (nothing, NN), (in, I..."


<br>

## Term Frequency-Inverse Document Frequency (TF-IDF)

---

- Calculate the TF-IDF values for words in your dataset. You can use an off-the-shelf method for this or come up with the formula yourself based on the information in the book.
- Add a new column ‘TF_IDF’ to your dataset containing the TF-IDF vector for each sentence.

In [8]:
def calculate_tfidf(df, max_features=100, min_df=1):
    """
    Calculate TF-IDF vectors for each sentence in the dataset.
    
    Args:
        df (pd.DataFrame): DataFrame containing column_name column
        max_features (int): Maximum number of features to keep
        min_df (int): Minimum document frequency for a term
        
    Returns:
        pd.DataFrame: Updated DataFrame with TF-IDF vectors
    """
    # Initialize TF-IDF vectorizer with adjusted parameters
    tfidf = TfidfVectorizer(
        max_features=max_features,
        min_df=min_df,           # Minimum document frequency
        lowercase=True,
        norm='l2',
        smooth_idf=True,
        strip_accents='unicode'  # Handle accented characters
    )
    
    # Fit and transform the sentences
    tfidf_matrix = tfidf.fit_transform(df[column_name].fillna(''))
    
    # Get feature names (words)
    feature_names = tfidf.get_feature_names_out()
    
    # Convert sparse matrix to dense and create DataFrame
    tfidf_dense = tfidf_matrix.todense()
    
    # Create dictionary representation for each row
    tfidf_vectors = []
    for i in range(tfidf_dense.shape[0]):
        # Get non-zero elements
        row = tfidf_dense[i].A1
        word_scores = {
            word: score 
            for word, score in zip(feature_names, row) 
            if score > 0
        }
        # If no scores above threshold, include the highest scoring term
        if not word_scores and len(row) > 0:
            max_idx = np.argmax(row)
            if row[max_idx] > 0:
                word_scores[feature_names[max_idx]] = row[max_idx]
        
        tfidf_vectors.append(word_scores)
    
    # Add TF-IDF vectors to DataFrame
    df['TF_IDF'] = tfidf_vectors
    
    return df

In [9]:
# Apply TF-IDF calculation
df = calculate_tfidf(df)
df

,text,emotion,POS_Tags,TF_IDF
0,Did you kiss?,surprise,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424..."
1,Yeah.,neutral,"[(yeah, NN), (., .)]",{'yeah': 1.0}
2,But no !,anger,"[(but, CC), (no, DT), (!, .)]","{'but': 0.6839659208399449, 'no': 0.7295139608..."
3,Why do you have a problem?,surprise,"[(why, WRB), (do, VBP), (you, PRP), (have, VBP...","{'do': 0.4891671774511191, 'have': 0.451626379..."
4,Did you kiss?,surprise,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424..."
...,...,...,...,...
24290,And tribes put everything on the line.,happiness,"[(and, CC), (tribes, VB), (put, VBN), (everyth...","{'and': 0.5237647476533076, 'on': 0.7210586798..."
24291,I came in claiming that I had the most knowled...,happiness,"[(i, NN), (came, VBD), (in, IN), (claiming, VB...","{'about': 0.4954202140000175, 'had': 0.5340731..."
24292,At no point during my time here was that dispr...,happiness,"[(at, IN), (no, DT), (point, NN), (during, IN)...","{'at': 0.39245326134474817, 'here': 0.39464254..."
24293,There is nothing in the last few days of my ex...,happiness,"[(there, EX), (is, VBZ), (nothing, NN), (in, I...","{'do': 0.32745236178764, 'in': 0.2929955349747..."


<br>

## Sentiment Analysis

---

- Perform sentiment analysis on each sentence using a library like TextBlob or VADER or another one that you find suiting for your data.
- Add a column ‘Sentiment_Score’ to your dataset with the sentiment polarity score for each sentence.

In [10]:
def analyze_sentiment(df, method='both'):
    """
    Perform sentiment analysis on text data using multiple methods.
    
    Args:
        df (pd.DataFrame): DataFrame containing column_name column
        method (str): 'textblob', 'vader', or 'both'
        
    Returns:
        pd.DataFrame: DataFrame with added sentiment columns
    """
    
    def get_textblob_sentiment(text):
        """Calculate TextBlob sentiment scores."""
        if not isinstance(text, str) or not text.strip():
            return {'polarity': 0.0, 'subjectivity': 0.0}
        
        try:
            analysis = TextBlob(text)
            return {
                'polarity': analysis.sentiment.polarity,
                'subjectivity': analysis.sentiment.subjectivity
            }
        except Exception as e:
            print(f"Error processing text with TextBlob: {e}")
            return {'polarity': 0.0, 'subjectivity': 0.0}

    def get_vader_sentiment(text):
        """Calculate VADER sentiment scores."""
        if not isinstance(text, str) or not text.strip():
            return {
                'compound': 0.0,
                'pos': 0.0,
                'neu': 0.0,
                'neg': 0.0
            }
        
        try:
            analyzer = SentimentIntensityAnalyzer()
            scores = analyzer.polarity_scores(text)
            return scores
        except Exception as e:
            print(f"Error processing text with VADER: {e}")
            return {
                'compound': 0.0,
                'pos': 0.0,
                'neu': 0.0,
                'neg': 0.0
            }

    def get_sentiment_label(score):
        """Convert numerical score to sentiment label."""
        if score > 0.05:
            return 'Positive'
        elif score < -0.05:
            return 'Negative'
        else:
            return 'Neutral'

    # Apply sentiment analysis based on method
    if method in ['textblob', 'both']:
        # TextBlob analysis
        textblob_results = df[column_name].apply(get_textblob_sentiment)
        df['TextBlob_Polarity'] = textblob_results.apply(lambda x: x['polarity'])
        df['TextBlob_Subjectivity'] = textblob_results.apply(lambda x: x['subjectivity'])
        df['TextBlob_Sentiment'] = df['TextBlob_Polarity'].apply(get_sentiment_label)

    if method in ['vader', 'both']:
        # VADER analysis
        vader_results = df[column_name].apply(get_vader_sentiment)
        df['VADER_Compound'] = vader_results.apply(lambda x: x['compound'])
        df['VADER_Positive'] = vader_results.apply(lambda x: x['pos'])
        df['VADER_Neutral'] = vader_results.apply(lambda x: x['neu'])
        df['VADER_Negative'] = vader_results.apply(lambda x: x['neg'])
        df['VADER_Sentiment'] = df['VADER_Compound'].apply(get_sentiment_label)

    # Create combined sentiment score if both methods are used
    if method == 'both':
        df['Sentiment_Score'] = (df['TextBlob_Polarity'] + df['VADER_Compound']) / 2
        df['Sentiment'] = df['Sentiment_Score'].apply(get_sentiment_label)

    # Add a readable summary column
    def get_sentiment_summary(row):
        """Create a readable summary of sentiment analysis."""
        summary = []
        if method in ['textblob', 'both']:
            summary.append(f"TextBlob: {row['TextBlob_Sentiment']} ({row['TextBlob_Polarity']:.2f})")
        if method in ['vader', 'both']:
            summary.append(f"VADER: {row['VADER_Sentiment']} ({row['VADER_Compound']:.2f})")
        if method == 'both':
            summary.append(f"Combined: {row['Sentiment']} ({row['Sentiment_Score']:.2f})")
        return ' | '.join(summary)

    # Add sentiment summary column
    df['Sentiment_Summary'] = df.apply(get_sentiment_summary, axis=1)

    # Only keep the "Sentiment_Score" and "Sentiment" columns (from the columns created in this function)
    df = df[[column_name, "POS_Tags", "TF_IDF", "Sentiment_Score", "Sentiment"]]
    
    return df

In [11]:
# Apply sentiment analysis
df = analyze_sentiment(df, method='both')
df

,text,POS_Tags,TF_IDF,Sentiment_Score,Sentiment
0,Did you kiss?,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424...",0.210750,Positive
1,Yeah.,"[(yeah, NN), (., .)]",{'yeah': 1.0},0.148000,Positive
2,But no !,"[(but, CC), (no, DT), (!, .)]","{'but': 0.6839659208399449, 'no': 0.7295139608...",-0.237650,Negative
3,Why do you have a problem?,"[(why, WRB), (do, VBP), (you, PRP), (have, VBP...","{'do': 0.4891671774511191, 'have': 0.451626379...",-0.200950,Negative
4,Did you kiss?,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424...",0.210750,Positive
...,...,...,...,...,...
24290,And tribes put everything on the line.,"[(and, CC), (tribes, VB), (put, VBN), (everyth...","{'and': 0.5237647476533076, 'on': 0.7210586798...",0.000000,Neutral
24291,I came in claiming that I had the most knowled...,"[(i, NN), (came, VBD), (in, IN), (claiming, VB...","{'about': 0.4954202140000175, 'had': 0.5340731...",0.025000,Neutral
24292,At no point during my time here was that dispr...,"[(at, IN), (no, DT), (point, NN), (during, IN)...","{'at': 0.39245326134474817, 'here': 0.39464254...",-0.148000,Negative
24293,There is nothing in the last few days of my ex...,"[(there, EX), (is, VBZ), (nothing, NN), (in, I...","{'do': 0.32745236178764, 'in': 0.2929955349747...",0.147267,Positive


<br>

## Pretrained Word Embeddings

---

- Use pretrained word embeddings (e.g., Word2Vec, GloVe) to convert sentences into vector representations.
- Add a column ‘Pretrained_Embeddings’ with the average word vector for each sentence.


In [12]:
class GloveEmbedder:
    """Class to handle sentence embeddings using GloVe word vectors."""
    
    def __init__(self, glove_path="./glove.840B.300d/glove.840B.300d.txt"):
        """Initialize the embedder with GloVe vectors."""
        print("Loading GloVe embeddings...")
        self.word_vectors = {}
        self.dim = 300  # GloVe 840B has 300 dimensions
        
        # Load GloVe word vectors
        with open(glove_path, 'r', encoding='utf-8') as f:
            for line in f:
                values = line.split(' ')
                word = values[0]
                vector = np.asarray(values[1:], dtype='float32')
                self.word_vectors[word] = vector
        
        print(f"Loaded {len(self.word_vectors)} word vectors")

    def get_embedding(self, sentence: str) -> np.ndarray:
        """
        Get GloVe embedding for a sentence by averaging word vectors.
        
        Args:
            sentence (str): Input sentence
            
        Returns:
            np.ndarray: Sentence embedding
        """
        if not isinstance(sentence, str):
            return np.zeros(self.dim)
            
        try:
            words = sentence.lower().split()
            vectors = [self.word_vectors.get(word, np.zeros(self.dim)) for word in words]
            if not vectors:
                return np.zeros(self.dim)
            return np.mean(vectors, axis=0)
        except Exception as e:
            print(f"Error encoding sentence: {e}")
            return np.zeros(self.dim)

    def embed_sentences(self, sentences: Union[List[str], pd.Series]) -> List[np.ndarray]:
        """
        Embed multiple sentences.
        
        Args:
            sentences: List or Series of sentences
            
        Returns:
            List[np.ndarray]: List of sentence embeddings
        """
        embeddings = []
        for sent in sentences:
            embedding = self.get_embedding(sent)
            embeddings.append(embedding.tolist())  # Convert to list to avoid numpy array issues
        return embeddings

def add_embeddings_to_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add embeddings to DataFrame.
    
    Args:
        df (pd.DataFrame): Input DataFrame with column_name column
        
    Returns:
        pd.DataFrame: DataFrame with added embedding columns
    """
    # Initialize embedder
    embedder = GloveEmbedder()
    
    # Create a copy of the dataframe to avoid SettingWithCopyWarning
    df = df.copy()
    
    # Get embeddings as a list
    embeddings = embedder.embed_sentences(df[column_name])
    
    # Assign embeddings using standard assignment
    df['Pretrained_Embeddings'] = embeddings
    
    # Add readable summary (first few dimensions)
    def format_vector(v, n=3):
        return f"[{', '.join(f'{x:.3f}' for x in v[:n])}...]"
    
    return df

# Apply embeddings to DataFrame
df = add_embeddings_to_df(df)
df

Loading GloVe embeddings...
Loaded 2196016 word vectors


,text,POS_Tags,TF_IDF,Sentiment_Score,Sentiment,Pretrained_Embeddings
0,Did you kiss?,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424...",0.210750,Positive,"[-0.05988466739654541, 0.23184999823570251, -0..."
1,Yeah.,"[(yeah, NN), (., .)]",{'yeah': 1.0},0.148000,Positive,"[0.15433000028133392, 0.1373399943113327, -0.4..."
2,But no !,"[(but, CC), (no, DT), (!, .)]","{'but': 0.6839659208399449, 'no': 0.7295139608...",-0.237650,Negative,"[-0.1412133425474167, 0.18132366240024567, -0...."
3,Why do you have a problem?,"[(why, WRB), (do, VBP), (you, PRP), (have, VBP...","{'do': 0.4891671774511191, 'have': 0.451626379...",-0.200950,Negative,"[-0.03720216608295838, 0.13633799677093825, -0..."
4,Did you kiss?,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424...",0.210750,Positive,"[-0.05988466739654541, 0.23184999823570251, -0..."
...,...,...,...,...,...,...
24290,And tribes put everything on the line.,"[(and, CC), (tribes, VB), (put, VBN), (everyth...","{'and': 0.5237647476533076, 'on': 0.7210586798...",0.000000,Neutral,"[-0.03757200017571449, -0.006746273022145033, ..."
24291,I came in claiming that I had the most knowled...,"[(i, NN), (came, VBD), (in, IN), (claiming, VB...","{'about': 0.4954202140000175, 'had': 0.5340731...",0.025000,Neutral,"[0.035636771470308304, 0.17163194715976715, -0..."
24292,At no point during my time here was that dispr...,"[(at, IN), (no, DT), (point, NN), (during, IN)...","{'at': 0.39245326134474817, 'here': 0.39464254...",-0.148000,Negative,"[-0.04344170084223151, 0.2609841007739305, -0...."
24293,There is nothing in the last few days of my ex...,"[(there, EX), (is, VBZ), (nothing, NN), (in, I...","{'do': 0.32745236178764, 'in': 0.2929955349747...",0.147267,Positive,"[0.09715829537633587, 0.18266641151379137, -0...."


<br>

## Custom Word Embedding Model

---

- Train your own word embedding model using your dataset with tools like Gensim’s Word2Vec.
- Add a column ‘Custom_Embeddings’ with the average word vector from your custom model for each sentence.


In [13]:
class CustomWordEmbedder:
    """Class to handle custom word embeddings using Word2Vec."""
    
    def __init__(self, vector_size=100, window=5, min_count=1):
        """
        Initialize the embedder with specified parameters.
        
        Args:
            vector_size (int): Dimensionality of word vectors
            window (int): Maximum distance between current and predicted word
            min_count (int): Minimum word frequency to include in vocabulary
        """
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.model = None
        
        # Download required NLTK data
        try:
            nltk.data.find('tokenizers/punkt')
        except LookupError:
            nltk.download('punkt')

    def preprocess_text(self, text: str) -> List[str]:
        """
        Preprocess text for training.
        
        Args:
            text (str): Input text
            
        Returns:
            List[str]: List of tokens
        """
        if not isinstance(text, str):
            return []
            
        # Tokenize and lowercase
        tokens = word_tokenize(text.lower(), language='english')
        
        # Basic cleaning (you can add more preprocessing steps)
        tokens = [
            token for token in tokens 
            if token.isalnum()  # Keep only alphanumeric
        ]
        
        return tokens

    def prepare_training_data(self, sentences: Union[List[str], pd.Series]) -> List[List[str]]:
        """
        Prepare data for training Word2Vec model.
        
        Args:
            sentences: List or Series of sentences
            
        Returns:
            List[List[str]]: List of tokenized sentences
        """
        return [
            self.preprocess_text(sent) 
            for sent in sentences 
            if isinstance(sent, str)
        ]

    def train_model(self, sentences: Union[List[str], pd.Series]):
        """
        Train Word2Vec model on the provided sentences.
        
        Args:
            sentences: List or Series of sentences
        """
        # Prepare training data
        training_data = self.prepare_training_data(sentences)
        
        # Train model
        print("Training Word2Vec model...")
        self.model = Word2Vec(
            sentences=training_data,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            workers=4  # Parallel processing
        )
        print("Model training completed!")

    def get_sentence_embedding(self, sentence: str) -> np.ndarray:
        """
        Get embedding for a sentence using trained model.
        
        Args:
            sentence (str): Input sentence
            
        Returns:
            np.ndarray: Average word vector for sentence
        """
        if not isinstance(sentence, str) or self.model is None:
            return np.zeros(self.vector_size)
            
        # Preprocess sentence
        tokens = self.preprocess_text(sentence)
        
        # Get vectors for words in vocabulary
        vectors = []
        for token in tokens:
            try:
                if token in self.model.wv:
                    vectors.append(self.model.wv[token])
            except KeyError:
                continue
        
        # Return average vector
        if vectors:
            return np.mean(vectors, axis=0)
        return np.zeros(self.vector_size)

    def embed_sentences(self, sentences: Union[List[str], pd.Series]) -> List[np.ndarray]:
        """
        Embed multiple sentences.
        
        Args:
            sentences: List or Series of sentences
            
        Returns:
            List[np.ndarray]: List of sentence embeddings
        """
        return [self.get_sentence_embedding(sent) for sent in sentences]

    def get_most_similar_words(self, word: str, n: int = 5) -> List[tuple]:
        """
        Get most similar words for a given word.
        
        Args:
            word (str): Input word
            n (int): Number of similar words to return
            
        Returns:
            List[tuple]: List of (word, similarity) pairs
        """
        if self.model is None or word not in self.model.wv:
            return []
            
        return self.model.wv.most_similar(word, topn=n)

In [14]:
def add_custom_embeddings_to_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add custom embeddings to DataFrame.
    
    Args:
        df (pd.DataFrame): Input DataFrame with column_name column
        
    Returns:
        pd.DataFrame: DataFrame with added embedding columns
    """
    # Initialize and train embedder
    embedder = CustomWordEmbedder()
    embedder.train_model(df[column_name])
    
    # Get embeddings
    df['Custom_Embeddings'] = embedder.embed_sentences(df[column_name])
    
    # Add readable summary
    def format_vector(v, n=3):
        return f"[{', '.join(f'{x:.3f}' for x in v[:n])}...]"
    
    # df['Custom_Embedding_Summary'] = df['Custom_Embeddings'].apply(format_vector)
    
    # Optional: Add vocabulary statistics
    print("\nVocabulary Statistics:")
    print(f"Total unique words: {len(embedder.model.wv)}")
    
    # Show example similar words
    print("\nExample similar words:")
    common_words = [word for word in embedder.model.wv.index_to_key[:10]]
    for word in common_words[:3]:  # Show for first 3 common words
        similar_words = embedder.get_most_similar_words(word)
        print(f"\nSimilar to '{word}':")
        for similar_word, similarity in similar_words:
            print(f"  {similar_word}: {similarity:.3f}")
    
    return df, embedder


# Apply custom embeddings to DataFrame
df, embedder = add_custom_embeddings_to_df(df)
df

Training Word2Vec model...
Model training completed!

Vocabulary Statistics:
Total unique words: 9082

Example similar words:

Similar to 'i':
  how: 0.979
  you: 0.979
  anything: 0.974
  why: 0.974
  what: 0.973

Similar to 'the':
  front: 0.981
  edge: 0.978
  island: 0.974
  top: 0.973
  road: 0.972

Similar to 'you':
  how: 0.980
  i: 0.979
  what: 0.978
  they: 0.976
  we: 0.976


,text,POS_Tags,TF_IDF,Sentiment_Score,Sentiment,Pretrained_Embeddings,Custom_Embeddings
0,Did you kiss?,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424...",0.210750,Positive,"[-0.05988466739654541, 0.23184999823570251, -0...","[-0.10402409, 0.5372206, -0.09934235, 0.200838..."
1,Yeah.,"[(yeah, NN), (., .)]",{'yeah': 1.0},0.148000,Positive,"[0.15433000028133392, 0.1373399943113327, -0.4...","[-0.23375244, 0.4365303, 0.10195672, 0.1710058..."
2,But no !,"[(but, CC), (no, DT), (!, .)]","{'but': 0.6839659208399449, 'no': 0.7295139608...",-0.237650,Negative,"[-0.1412133425474167, 0.18132366240024567, -0....","[-0.47401512, 0.6390312, -0.0054327548, 0.3530..."
3,Why do you have a problem?,"[(why, WRB), (do, VBP), (you, PRP), (have, VBP...","{'do': 0.4891671774511191, 'have': 0.451626379...",-0.200950,Negative,"[-0.03720216608295838, 0.13633799677093825, -0...","[-0.15472351, 0.6196188, -0.028828641, 0.25967..."
4,Did you kiss?,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424...",0.210750,Positive,"[-0.05988466739654541, 0.23184999823570251, -0...","[-0.10402409, 0.5372206, -0.09934235, 0.200838..."
...,...,...,...,...,...,...,...
24290,And tribes put everything on the line.,"[(and, CC), (tribes, VB), (put, VBN), (everyth...","{'and': 0.5237647476533076, 'on': 0.7210586798...",0.000000,Neutral,"[-0.03757200017571449, -0.006746273022145033, ...","[-0.14447534, 0.6020859, 0.16885749, 0.0555446..."
24291,I came in claiming that I had the most knowled...,"[(i, NN), (came, VBD), (in, IN), (claiming, VB...","{'about': 0.4954202140000175, 'had': 0.5340731...",0.025000,Neutral,"[0.035636771470308304, 0.17163194715976715, -0...","[-0.17922348, 0.59568506, 0.1301484, 0.1875022..."
24292,At no point during my time here was that dispr...,"[(at, IN), (no, DT), (point, NN), (during, IN)...","{'at': 0.39245326134474817, 'here': 0.39464254...",-0.148000,Negative,"[-0.04344170084223151, 0.2609841007739305, -0....","[-0.28680775, 0.59227854, 0.09806119, 0.187424..."
24293,There is nothing in the last few days of my ex...,"[(there, EX), (is, VBZ), (nothing, NN), (in, I...","{'do': 0.32745236178764, 'in': 0.2929955349747...",0.147267,Positive,"[0.09715829537633587, 0.18266641151379137, -0....","[-0.24863593, 0.65601015, 0.16513708, 0.177164..."


<br>

## Additional Feature

---

- Extract at least one more additional NLP feature.
- Add a column (using the same naming convention ‘Xxx_Xxx’) for each sentence.

In [17]:
class TextAnalyzer:
    """Class to handle advanced text analysis features."""
    
    def __init__(self, language='english'):
        """
        Initialize the analyzer with required models.
        
        Args:
            language (str): Language code ('english', 'spanish', etc.)
        """
        self.language = language.lower()
        print(f"Loading language models for {language}...")
        
        # Map language codes to spaCy models
        spacy_models = {
            'english': 'en_core_web_lg',
            'spanish': 'es_core_news_lg',
            'french': 'fr_core_news_lg',
            'german': 'de_core_news_lg',
        }
        
        try:
            model_name = spacy_models.get(self.language, 'en_core_web_lg')
            self.nlp = spacy.load(model_name)
        except OSError:
            print(f"SpaCy model for {language} not found. Using English model.")
            self.nlp = spacy.load('en_core_web_lg')
        
        # Get stop words for the specified language
        try:
            self.stop_words = set(stopwords.words(language))
        except:
            print(f"Stop words not available for {language}. Using English stop words.")
            self.stop_words = set(stopwords.words('english'))
    
    def calculate_complexity_metrics(self, text):
        """Calculate various text complexity metrics."""
        if not isinstance(text, str):
            return {
                'avg_word_length': 0,
                'unique_words_ratio': 0,
                'lexical_density': 0
            }
            
        # Process with spaCy
        doc = self.nlp(text)
        
        # Get words based on language-specific tokenization
        words = [token.text.lower() for token in doc if token.is_alpha]
        content_words = [
            token.text.lower() 
            for token in doc 
            if token.is_alpha and token.text.lower() not in self.stop_words
        ]
        
        if not words:
            return {
                'avg_word_length': 0,
                'unique_words_ratio': 0,
                'lexical_density': 0
            }
        
        metrics = {
            'avg_word_length': np.mean([len(word) for word in words]),
            'unique_words_ratio': len(set(words)) / len(words),
            'lexical_density': len(content_words) / len(words)
        }
        
        return metrics
    
    def extract_text_patterns(self, text: str) -> Dict[str, Any]:
        """
        Extract various text patterns and statistics.
        
        Args:
            text (str): Input text
            
        Returns:
            dict: Dictionary of text patterns and statistics
        """
        if not isinstance(text, str):
            return {
                'question_marks': 0,
                'exclamation_marks': 0,
                'capitalized_words': 0,
                'numerics': 0
            }
            
        patterns = {
            'question_marks': text.count('?'),
            'exclamation_marks': text.count('!'),
            'capitalized_words': len(re.findall(r'\b[A-Z][a-z]+\b', text)),
            'numerics': len(re.findall(r'\d+', text))
        }
        
        return patterns

In [18]:
def add_nlp_features(df: pd.DataFrame, language='english') -> pd.DataFrame:
    """
    Add additional NLP features to DataFrame.
    
    Args:
        df (pd.DataFrame): Input DataFrame with column_name column
        language (str): Language code ('english', 'spanish', etc.)
        
    Returns:
        pd.DataFrame: DataFrame with added NLP features
    """
    # Initialize analyzer with specified language
    analyzer = TextAnalyzer(language=language)
    
    # Calculate complexity metrics
    complexity_metrics = df[column_name].apply(analyzer.calculate_complexity_metrics)
    df['Avg_Word_Length'] = complexity_metrics.apply(lambda x: x['avg_word_length'])
    df['Lexical_Density'] = complexity_metrics.apply(lambda x: x['lexical_density'])
    
    # Extract text patterns
    text_patterns = df[column_name].apply(analyzer.extract_text_patterns)
    df['Exclamation_Marks'] = text_patterns.apply(lambda x: x['exclamation_marks'])
    df['Capitalized_Words'] = text_patterns.apply(lambda x: x['capitalized_words'])
    df['Numerics'] = text_patterns.apply(lambda x: x['numerics'])
    
    return df

# Apply additional NLP features
df = add_nlp_features(df)

df

Loading language models for english...


,text,POS_Tags,TF_IDF,Sentiment_Score,Sentiment,Pretrained_Embeddings,Custom_Embeddings,Avg_Word_Length,Lexical_Density,Exclamation_Marks,Capitalized_Words,Numerics
0,Did you kiss?,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424...",0.210750,Positive,"[-0.05988466739654541, 0.23184999823570251, -0...","[-0.10402409, 0.5372206, -0.09934235, 0.200838...",3.333333,0.333333,0,1,0
1,Yeah.,"[(yeah, NN), (., .)]",{'yeah': 1.0},0.148000,Positive,"[0.15433000028133392, 0.1373399943113327, -0.4...","[-0.23375244, 0.4365303, 0.10195672, 0.1710058...",4.000000,1.000000,0,1,0
2,But no !,"[(but, CC), (no, DT), (!, .)]","{'but': 0.6839659208399449, 'no': 0.7295139608...",-0.237650,Negative,"[-0.1412133425474167, 0.18132366240024567, -0....","[-0.47401512, 0.6390312, -0.0054327548, 0.3530...",2.500000,0.000000,1,1,0
3,Why do you have a problem?,"[(why, WRB), (do, VBP), (you, PRP), (have, VBP...","{'do': 0.4891671774511191, 'have': 0.451626379...",-0.200950,Negative,"[-0.03720216608295838, 0.13633799677093825, -0...","[-0.15472351, 0.6196188, -0.028828641, 0.25967...",3.333333,0.166667,0,1,0
4,Did you kiss?,"[(did, VBD), (you, PRP), (kiss, VB), (?, .)]","{'did': 0.8866279274682668, 'you': 0.462483424...",0.210750,Positive,"[-0.05988466739654541, 0.23184999823570251, -0...","[-0.10402409, 0.5372206, -0.09934235, 0.200838...",3.333333,0.333333,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
24290,And tribes put everything on the line.,"[(and, CC), (tribes, VB), (put, VBN), (everyth...","{'and': 0.5237647476533076, 'on': 0.7210586798...",0.000000,Neutral,"[-0.03757200017571449, -0.006746273022145033, ...","[-0.14447534, 0.6020859, 0.16885749, 0.0555446...",4.428571,0.571429,0,1,0
24291,I came in claiming that I had the most knowled...,"[(i, NN), (came, VBD), (in, IN), (claiming, VB...","{'about': 0.4954202140000175, 'had': 0.5340731...",0.025000,Neutral,"[0.035636771470308304, 0.17163194715976715, -0...","[-0.17922348, 0.59568506, 0.1301484, 0.1875022...",3.923077,0.307692,0,0,0
24292,At no point during my time here was that dispr...,"[(at, IN), (no, DT), (point, NN), (during, IN)...","{'at': 0.39245326134474817, 'here': 0.39464254...",-0.148000,Negative,"[-0.04344170084223151, 0.2609841007739305, -0....","[-0.28680775, 0.59227854, 0.09806119, 0.187424...",4.100000,0.300000,0,1,0
24293,There is nothing in the last few days of my ex...,"[(there, EX), (is, VBZ), (nothing, NN), (in, I...","{'do': 0.32745236178764, 'in': 0.2929955349747...",0.147267,Positive,"[0.09715829537633587, 0.18266641151379137, -0....","[-0.24863593, 0.65601015, 0.16513708, 0.177164...",4.235294,0.352941,0,2,0


<br>

## Save the File

---

In [19]:
df.to_excel('NLP_features.xlsx', index=False)